# 03 - Generative VAE: Pretrain, Finetune, and Visualization (NaN/Inf Fix Applied)

**Note**: This notebook includes automated NaN/Inf detection and sanitization before pretraining to prevent batch skips.

# 03 - Generative VAE: Pretrain, Finetune, and Visualization

This notebook demonstrates a lightweight two-phase VAE workflow: a short pretraining pass (ZINC-style), followed by a finetune pass on target-like data. For demo purposes this runs on synthetic data and produces an RDKit grid image of example molecules.

In [1]:
%pip install pandas numpy matplotlib seaborn scikit-learn torch-geometric rdkit-pypi tensorboard biopython

# Setup and imports
# Resolve the project root up front so the pretraining and fine-tuning phases can reuse the same data and model paths.
import sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
import torch
import numpy as np
from core.vae_architecture import GraphVAE

# These helpers keep the notebook runnable on both GPU and Apple Silicon, and normalize graph shapes before batching.
def get_working_device():
    if torch.cuda.is_available():
        try:
            torch.zeros(1).to('cuda')
            return torch.device('cuda')
        except Exception:
            pass
    if torch.backends.mps.is_available():
        try:
            torch.zeros(1).to('mps')
            return torch.device('mps')
        except Exception:
            pass
    return torch.device('cpu')

def normalize_edge_index(data):
    edge_index = getattr(data, 'edge_index', None)
    if edge_index is None:
        data.edge_index = torch.zeros((2, 0), dtype=torch.long)
        return data
    if not hasattr(edge_index, 'ndim') or edge_index.ndim != 2:
        data.edge_index = torch.zeros((2, 0), dtype=torch.long)
        return data
    if 0 in edge_index.shape:
        data.edge_index = torch.zeros((2, 0), dtype=torch.long)
        return data
    if edge_index.shape[0] == 2:
        return data
    if edge_index.shape[1] == 2:
        data.edge_index = edge_index.t().contiguous()
        return data
    data.edge_index = torch.zeros((2, 0), dtype=torch.long)
    return data

def truncate_graph_edges(data, max_nodes):
    edge_index = getattr(data, 'edge_index', None)
    edge_attr = getattr(data, 'edge_attr', None)
    if edge_index is None or not hasattr(edge_index, 'ndim') or edge_index.ndim != 2 or edge_index.shape[1] == 0:
        data.edge_index = torch.zeros((2, 0), dtype=torch.long)
        if edge_attr is not None:
            data.edge_attr = edge_attr.new_zeros((0, edge_attr.shape[-1]))
        return data
    if edge_index.shape[0] == 2:
        node_mask = (edge_index[0] < max_nodes) & (edge_index[1] < max_nodes)
    else:
        node_mask = (edge_index[:, 0] < max_nodes) & (edge_index[:, 1] < max_nodes)
        edge_index = edge_index.t().contiguous()
    edge_index = edge_index[:, node_mask]
    if edge_attr is not None and hasattr(edge_attr, 'shape') and edge_attr.shape[0] == node_mask.shape[0]:
        edge_attr = edge_attr[node_mask]
    if edge_index.numel() == 0:
        data.edge_index = torch.zeros((2, 0), dtype=torch.long)
        if edge_attr is not None:
            data.edge_attr = edge_attr.new_zeros((0, edge_attr.shape[-1]))
        return data
    data.edge_index = edge_index
    if edge_attr is not None:
        data.edge_attr = edge_attr
    return data
OUT = ROOT / 'results' / 'figures'
OUT.mkdir(parents=True, exist_ok=True)
device = get_working_device()
print(f'Ready; device={device}')

ERROR: Could not find a version that satisfies the requirement rdkit-pypi (from versions: none)
ERROR: No matching distribution found for rdkit-pypi


Note: you may need to restart the kernel to use updated packages.


c:\Users\u2251865\.conda\envs\tree4\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ready; device=cuda


In [2]:
# ===== PHASE 1: VAE PRETRAINING ON REAL ZINC DATA =====

DATA_PROC = ROOT / 'data' / 'processed'
DATA_RAW = ROOT / 'data' / 'raw'

print('='*60)
print('PHASE 1: Pre-training on ZINC (Real Large-Scale Data)')
print('='*60)

# Load SMILES from ZINC (either .smi or .csv)
zinc_smiles = []
zinc_file_smi = DATA_RAW / 'zinc250k.smi'
zinc_file_csv = DATA_RAW / 'zinc250k.csv'

if zinc_file_smi.exists():
    print(f'\nLoading ZINC from {zinc_file_smi.name}...')
    with open(zinc_file_smi, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                zinc_smiles.append(parts[0])
    print(f'  Loaded {len(zinc_smiles)} SMILES')
elif zinc_file_csv.exists():
    print(f'\nLoading ZINC from {zinc_file_csv.name}...')
    import pandas as pd
    zinc_df = pd.read_csv(zinc_file_csv)
    zinc_smiles = zinc_df['smiles'].astype(str).tolist()
    print(f'  Loaded {len(zinc_smiles)} SMILES')
else:
    print('ZINC file not found; using synthetic data fallback')
    zinc_smiles = None

if zinc_smiles:
    n_pretrain = min(200000, len(zinc_smiles))
    zinc_smiles = zinc_smiles[:n_pretrain]
    print(f'Using {len(zinc_smiles)} ZINC molecules for pretraining')
else:
    print('(Pretraining phase will skip real data)')

from rdkit import Chem
from rdkit.Chem import AllChem
from torch_geometric.data import Data as PyGData


def safe_float(val, default=0.0):
    try:
        out = float(val)
    except Exception:
        return default
    if not np.isfinite(out):
        return default
    return out


def get_node_features(atom):
    partial_charge = 0.0
    if atom.HasProp('_GasteigerCharge'):
        partial_charge = safe_float(atom.GetProp('_GasteigerCharge'), default=0.0)
    partial_charge = float(np.clip(partial_charge, -5.0, 5.0))

    return [
        float(atom.GetAtomicNum()),
        float(atom.GetTotalDegree()),
        float(atom.GetFormalCharge()),
        float(int(atom.GetHybridization())),
        float(int(atom.GetIsAromatic())),
        float(atom.GetMass()),
        float(atom.GetTotalNumHs()),
        partial_charge,
    ]


def get_edge_features(bond):
    bond_type = bond.GetBondType()
    return [
        int(bond_type == Chem.rdchem.BondType.SINGLE),
        int(bond_type == Chem.rdchem.BondType.DOUBLE),
        int(bond_type == Chem.rdchem.BondType.TRIPLE),
        int(bond_type == Chem.rdchem.BondType.AROMATIC),
    ]


def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    try:
        AllChem.ComputeGasteigerCharges(mol)
    except Exception:
        pass

    x = [get_node_features(atom) for atom in mol.GetAtoms()]
    edge_index = [[], []]
    edge_attr = []

    for bond in mol.GetBonds():
        a1, a2 = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_index[0].extend([a1, a2])
        edge_index[1].extend([a2, a1])
        features = get_edge_features(bond)
        edge_attr.extend([features, features])

    x_t = torch.tensor(x, dtype=torch.float)
    x_t = torch.nan_to_num(x_t, nan=0.0, posinf=1e4, neginf=-1e4).clamp(-1e4, 1e4)

    if len(edge_attr) == 0:
        edge_attr_t = torch.zeros((0, 4), dtype=torch.float)
        edge_index_t = torch.zeros((2, 0), dtype=torch.long)
    else:
        edge_attr_t = torch.tensor(edge_attr, dtype=torch.float)
        edge_attr_t = torch.nan_to_num(edge_attr_t, nan=0.0, posinf=1.0, neginf=0.0)
        edge_index_t = torch.tensor(edge_index, dtype=torch.long)

    data = PyGData(x=x_t, edge_index=edge_index_t, edge_attr=edge_attr_t)
    return data


def standardize_feature_tensor(features, eps=1e-6):
    if features is None or features.numel() == 0:
        return features
    feature_mean = features.mean(dim=0, keepdim=True)
    feature_std = features.std(dim=0, keepdim=True, unbiased=False).clamp_min(eps)
    standardized = (features - feature_mean) / feature_std
    return torch.nan_to_num(standardized, nan=0.0, posinf=0.0, neginf=0.0)


print('\nConverting ZINC SMILES to PyG graphs...')
zinc_graphs = []
if zinc_smiles:
    for i, smiles in enumerate(zinc_smiles):
        if i % 5000 == 0:
            print(f'  Processed {i}/{len(zinc_smiles)}')
        g = smiles_to_graph(smiles)
        if g is not None:
            zinc_graphs.append(g)

    zinc_graphs = [normalize_edge_index(g) for g in zinc_graphs]

    # Dataset-wide sanitization pass
    fixed_graphs = 0
    for g in zinc_graphs:
        changed = False
        if torch.isnan(g.x).any() or torch.isinf(g.x).any():
            g.x = torch.nan_to_num(g.x, nan=0.0, posinf=1e4, neginf=-1e4).clamp(-1e4, 1e4)
            changed = True
        if getattr(g, 'edge_attr', None) is not None and (torch.isnan(g.edge_attr).any() or torch.isinf(g.edge_attr).any()):
            g.edge_attr = torch.nan_to_num(g.edge_attr, nan=0.0, posinf=1.0, neginf=0.0)
            changed = True
        if changed:
            fixed_graphs += 1

    print(f'Converted {len(zinc_graphs)} valid ZINC graphs')
    print(f'Sanitization pass fixed {fixed_graphs} graphs')
else:
    print('(Skipping graph conversion: using synthetic fallback)')

edge_feat_dim = int(zinc_graphs[0].edge_attr.shape[1]) if zinc_graphs and getattr(zinc_graphs[0], 'edge_attr', None) is not None else 4
print(f'Inferred ZINC edge feature width: {edge_feat_dim}')

max_nodes = 50
epochs_pre = 15

vae = GraphVAE(
    node_features=8,
    edge_features=edge_feat_dim,
    hidden_dim=128,
    latent_dim=64,
    max_nodes=max_nodes,
    use_pocket_conditioning=False,
)
vae = vae.to(device)

optimizer = torch.optim.Adam(vae.parameters(), lr=5e-4)
criterion_recon = torch.nn.MSELoss()

print('\nTraining VAE pretraining phase')
print(f'  Graphs: {len(zinc_graphs)}, Epochs: {epochs_pre}, Max nodes: {max_nodes}')
print('  (Processing in mini-batches of 32 graphs)\n')

vae.train()

if len(zinc_graphs) > 0:
    batch_size = 32
    n_batches = (len(zinc_graphs) + batch_size - 1) // batch_size

    for ep in range(epochs_pre):
        total_loss = 0.0
        n_processed = 0
        skipped_batches = 0

        for batch_idx in range(n_batches):
            start_idx = batch_idx * batch_size
            end_idx = min(start_idx + batch_size, len(zinc_graphs))
            batch_graphs = zinc_graphs[start_idx:end_idx]

            if len(batch_graphs) == 0:
                continue

            optimizer.zero_grad()

            batch_x_list = []
            batch_edge_index_list = []
            batch_edge_attr_list = []
            batch_indices = []
            node_offset = 0

            for g_idx, g in enumerate(batch_graphs):
                x = g.x
                n_atoms = x.shape[0]

                if n_atoms < max_nodes:
                    x = torch.cat([x, torch.zeros(max_nodes - n_atoms, x.shape[1], device=x.device, dtype=x.dtype)], dim=0)
                else:
                    x = x[:max_nodes]

                x = torch.nan_to_num(x, nan=0.0, posinf=1e4, neginf=-1e4).clamp(-1e4, 1e4)
                batch_x_list.append(x)

                g = truncate_graph_edges(g, max_nodes)
                if g.edge_index.shape[1] > 0:
                    edge_idx = g.edge_index + node_offset
                    batch_edge_index_list.append(edge_idx)
                    ea = torch.nan_to_num(g.edge_attr, nan=0.0, posinf=1.0, neginf=0.0)
                    batch_edge_attr_list.append(ea)

                batch_indices.extend([g_idx] * max_nodes)
                node_offset += max_nodes

            X_batch = torch.cat(batch_x_list, dim=0)

            if batch_edge_index_list:
                edge_index_batch = torch.cat(batch_edge_index_list, dim=1)
                edge_attr_batch = torch.cat(batch_edge_attr_list, dim=0)
            else:
                edge_index_batch = torch.zeros((2, 0), dtype=torch.long)
                edge_attr_batch = torch.zeros((0, edge_feat_dim), dtype=torch.float)

            batch_tensor = torch.tensor(batch_indices, dtype=torch.long)
            global_feat = standardize_feature_tensor(torch.zeros(len(batch_graphs), 2))

            X_batch = X_batch.to(device)
            edge_index_batch = edge_index_batch.to(device)
            edge_attr_batch = edge_attr_batch.to(device)
            batch_tensor = batch_tensor.to(device)
            global_feat = global_feat.to(device)

            if not torch.isfinite(X_batch).all() or not torch.isfinite(edge_attr_batch).all():
                skipped_batches += 1
                print(f'Non-finite input tensors at epoch {ep+1}, batch {batch_idx+1}; skipping batch')
                continue

            mu, logvar = vae.encode(X_batch, edge_index_batch, edge_attr_batch, batch_tensor, global_feat)

            if not torch.isfinite(mu).all() or not torch.isfinite(logvar).all():
                skipped_batches += 1
                print(f'Non-finite encode output at epoch {ep+1}, batch {batch_idx+1}; skipping batch')
                continue

            mu = torch.clamp(mu, -30.0, 30.0)
            logvar = torch.clamp(logvar, -20.0, 20.0)

            z = vae.reparameterize(mu, logvar)
            node_logits, edge_adj, edge_type = vae.decode(z, pocket_embedding=None)

            if not torch.isfinite(node_logits).all() or not torch.isfinite(edge_adj).all() or not torch.isfinite(edge_type).all():
                skipped_batches += 1
                print(f'Non-finite decode output at epoch {ep+1}, batch {batch_idx+1}; skipping batch')
                continue

            X_reshaped = X_batch.view(len(batch_graphs), max_nodes, -1)
            loss_recon = criterion_recon(node_logits, X_reshaped)
            kld = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

            beta = 0.01 * (ep + 1) / epochs_pre
            loss = loss_recon + beta * kld

            if torch.isnan(loss) or not torch.isfinite(loss):
                skipped_batches += 1
                print(f'NaN/Inf loss at epoch {ep+1}, batch {batch_idx+1}; skipping batch')
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(vae.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item() * len(batch_graphs)
            n_processed += len(batch_graphs)

        avg_loss = total_loss / max(n_processed, 1)
        print(f'Epoch {ep+1}/{epochs_pre} - Loss: {avg_loss:.4f}, beta: {beta:.4f}, skipped: {skipped_batches}')
else:
    print('No ZINC graphs available; skipping pretraining')

models_dir = ROOT / 'models'
models_dir.mkdir(exist_ok=True, parents=True)
torch.save(vae.state_dict(), models_dir / 'zinc_pretrained_graphvae.pth')
print(f'\nPretrained VAE saved: {models_dir / "zinc_pretrained_graphvae.pth"}')

PHASE 1: Pre-training on ZINC (Real Large-Scale Data)

Loading ZINC from zinc250k.csv...
  Loaded 249455 SMILES
Using 200000 ZINC molecules for pretraining

Converting ZINC SMILES to PyG graphs...
  Processed 0/200000
  Processed 5000/200000
  Processed 10000/200000
  Processed 15000/200000
  Processed 20000/200000
  Processed 25000/200000
  Processed 30000/200000
  Processed 35000/200000
  Processed 40000/200000
  Processed 45000/200000
  Processed 50000/200000
  Processed 55000/200000
  Processed 60000/200000
  Processed 65000/200000
  Processed 70000/200000
  Processed 75000/200000
  Processed 80000/200000
  Processed 85000/200000
  Processed 90000/200000
  Processed 95000/200000
  Processed 100000/200000
  Processed 105000/200000
  Processed 110000/200000
  Processed 115000/200000
  Processed 120000/200000
  Processed 125000/200000
  Processed 130000/200000
  Processed 135000/200000
  Processed 140000/200000
  Processed 145000/200000
  Processed 150000/200000
  Processed 155000/200

In [3]:
# ===== PHASE 2: FINE-TUNING ON REAL EGFR DATA WITH POCKET CONDITIONING =====
print('\n' + '='*60)
print('PHASE 2: Fine-tuning on EGFR (With Pocket Conditioning)')
print('='*60)

from collections import Counter
import torch.nn.functional as F
from core.vae_architecture import GraphVAE
from core.pocket_extractor import get_pocket_embedding

device = get_working_device()
print(f'Using device for fine-tuning: {device}')

DATA_PROC = ROOT / 'data' / 'processed'
DATA_RAW = ROOT / 'data' / 'raw'


def standardize_feature_tensor(features, eps=1e-6):
    if features is None or features.numel() == 0:
        return features
    feature_mean = features.mean(dim=0, keepdim=True)
    feature_std = features.std(dim=0, keepdim=True, unbiased=False).clamp_min(eps)
    standardized = (features - feature_mean) / feature_std
    return torch.nan_to_num(standardized, nan=0.0, posinf=0.0, neginf=0.0)

graph_file = DATA_PROC / 'graph_data.pt'
if not graph_file.exists():
    print(f'Missing graph bundle: {graph_file}')
    egfr_graphs = []
else:
    loaded_payload = torch.load(graph_file, weights_only=False)
    if isinstance(loaded_payload, dict):
        graph_candidates = [loaded_payload.get('graphs'), loaded_payload.get('egfr_graphs'), loaded_payload.get('train_graphs'), loaded_payload.get('data')]
        egfr_graphs = []
        for candidate in graph_candidates:
            if isinstance(candidate, (list, tuple)) and len(candidate) > 0:
                egfr_graphs = list(candidate)
                break
        if len(egfr_graphs) == 0:
            for candidate in loaded_payload.values():
                if isinstance(candidate, (list, tuple)) and len(candidate) > 0:
                    egfr_graphs = list(candidate)
                    break
    elif isinstance(loaded_payload, (list, tuple)):
        egfr_graphs = list(loaded_payload)
    else:
        egfr_graphs = []

    egfr_graphs = [g for g in egfr_graphs if hasattr(g, 'x')]
    egfr_graphs = [normalize_edge_index(g) for g in egfr_graphs]
    print(f'Loaded {len(egfr_graphs)} EGFR graphs from {graph_file.name}')

if len(egfr_graphs) == 0:
    print('No EGFR graphs available; skipping fine-tuning.')
else:
    sample_graph = egfr_graphs[0]
    node_feature_dim = int(sample_graph.x.shape[1]) if getattr(sample_graph, 'x', None) is not None else 8
    edge_feature_dim = int(sample_graph.edge_attr.shape[1]) if getattr(sample_graph, 'edge_attr', None) is not None and sample_graph.edge_attr.numel() > 0 else 4

    class_counter = Counter()
    for g in egfr_graphs:
        if getattr(g, 'x', None) is None:
            continue
        atomic_nums = g.x[:, 0].to(torch.long).tolist()
        for atomic_num in atomic_nums:
            if atomic_num == 6:
                class_counter['C'] += 1
            elif atomic_num == 7:
                class_counter['N'] += 1
            elif atomic_num == 8:
                class_counter['O'] += 1
            elif atomic_num == 9:
                class_counter['F'] += 1
            elif atomic_num == 16:
                class_counter['S'] += 1
            elif atomic_num == 17:
                class_counter['Cl'] += 1
            else:
                class_counter['C'] += 1

    atom_symbols = ['C', 'N', 'O', 'F', 'S', 'Cl']
    class_counts = torch.tensor([class_counter.get(sym, 0) for sym in atom_symbols], dtype=torch.float32).clamp_min(1.0)
    class_weights = (1.0 / class_counts)
    class_weights = class_weights / class_weights.sum() * len(atom_symbols)
    class_weights = class_weights.clamp(max=1.5).to(device)
    
    # --- NEW CODE: SUPERVISOR HETEROATOM BOOST ---
    # Indices: 0=C, 1=N, 2=O, 3=F, 4=S, 5=Cl
    class_weights[1] *= 5.0  # Boost Nitrogen weight 5x
    class_weights[2] *= 10.0 # Boost Oxygen weight 10x
    # ---------------------------------------------

    print(f'Atom class counts: {class_counts.tolist()}')
    print(f'Atom class weights (boosted): {class_weights.tolist()}')

    pocket_dim = 128
    pdb_file = DATA_RAW / 'pdb' / '3W2S.pdb'
    real_pocket_array = get_pocket_embedding(
        str(pdb_file),
        ligand_code='W2R',
        pocket_radius=7.0,
        embedding_dim=pocket_dim,
    )
    real_pocket = torch.tensor(real_pocket_array, dtype=torch.float32, device=device).unsqueeze(0)
    print(f'Using pocket embedding shape: {tuple(real_pocket.shape)}')

    pretrained_path = ROOT / 'models' / 'zinc_pretrained_graphvae.pth'
    vae = GraphVAE(
        node_features=node_feature_dim,
        edge_features=edge_feature_dim,
        global_features=2,
        hidden_dim=128,
        latent_dim=64,
        max_nodes=50,
        pocket_dim=pocket_dim,
        use_pocket_conditioning=True,
    ).to(device)

    if pretrained_path.exists():
        pretrained_state = torch.load(pretrained_path, map_location=device)
        current_state = vae.state_dict()
        compatible_state = {
            key: value
            for key, value in pretrained_state.items()
            if key in current_state and current_state[key].shape == value.shape
        }
        current_state.update(compatible_state)
        vae.load_state_dict(current_state)
        print(f'Loaded pretrained weights from {pretrained_path.name} with pocket_dim={pocket_dim}')
    else:
        print(f'Pretrained weights not found at {pretrained_path}; training from scratch')

    for parameter in vae.encoder.parameters():
        parameter.requires_grad = False

    if 'atom_classifier' not in globals() or atom_classifier.in_features != vae.decoder.node_features or atom_classifier.out_features != len(atom_symbols):
        atom_classifier = torch.nn.Linear(vae.decoder.node_features, len(atom_symbols)).to(device)
    else:
        atom_classifier = atom_classifier.to(device)

    trainable_params = list(filter(lambda p: p.requires_grad, vae.parameters())) + list(atom_classifier.parameters())
    optimizer = torch.optim.Adam(trainable_params, lr=1e-3)
    criterion_recon = torch.nn.MSELoss()
    edge_adj_criterion = torch.nn.BCEWithLogitsLoss()

    epochs_ft = 5
    batch_size = 16
    max_nodes = 50
    alpha_atom = 1.0
    lambda_halogen = 0.1
    lambda_edge = 0.5
    beta = 0.005
    tau_start = 1.0
    tau_end = 0.5
    tau_decay_epochs = 5
    expected_edge_dim = edge_feature_dim
    halogen_idxs = [atom_symbols.index('F'), atom_symbols.index('Cl')]
    n_batches = (len(egfr_graphs) + batch_size - 1) // batch_size

    print(f'\nFine-tuning for {epochs_ft} epochs on {len(egfr_graphs)} EGFR graphs...')
    print(f'Expected edge feature dim: {expected_edge_dim}')

    vae.train()
    atom_classifier.train()

    for ep in range(epochs_ft):
        warmup_scale = min(1.0, (ep + 1) / 5.0)
        for param_group in optimizer.param_groups:
            param_group['lr'] = 1e-3 * warmup_scale

        total_loss = 0.0
        total_ce = 0.0
        total_halogen_pen = 0.0
        total_edge_adj = 0.0
        total_edge_type = 0.0
        n_processed = 0
        skipped_batches = 0
        tau_progress = min(ep + 1, tau_decay_epochs) / tau_decay_epochs
        tau = tau_start + (tau_end - tau_start) * tau_progress

        for batch_idx in range(n_batches):
            start_idx = batch_idx * batch_size
            end_idx = min(start_idx + batch_size, len(egfr_graphs))
            batch_graphs = egfr_graphs[start_idx:end_idx]
            if len(batch_graphs) == 0:
                continue

            optimizer.zero_grad()
            batch_x_list = []
            batch_edge_index_list = []
            batch_edge_attr_list = []
            batch_adj_targets = []
            batch_type_targets = []
            batch_indices = []
            node_offset = 0

            for g_idx, g in enumerate(batch_graphs):
                g = truncate_graph_edges(g, max_nodes)
                x = g.x
                n_atoms = int(x.shape[0])
                if n_atoms < max_nodes:
                    x = torch.cat([x, torch.zeros(max_nodes - n_atoms, x.shape[1], device=x.device, dtype=x.dtype)], dim=0)
                else:
                    x = x[:max_nodes]
                x = torch.nan_to_num(x, nan=0.0, posinf=1e4, neginf=-1e4).clamp(-1e4, 1e4)
                batch_x_list.append(x)

                adj_target = torch.zeros((max_nodes, max_nodes), dtype=torch.float32)
                type_target = torch.full((max_nodes, max_nodes), -100, dtype=torch.long)

                if getattr(g, 'edge_index', None) is not None and g.edge_index.shape[1] > 0:
                    edge_index = g.edge_index
                    edge_attr = getattr(g, 'edge_attr', None)
                    if edge_attr is None or edge_attr.numel() == 0:
                        edge_labels = torch.zeros(edge_index.shape[1], dtype=torch.long)
                        edge_attr = torch.zeros((edge_index.shape[1], expected_edge_dim), dtype=torch.float32)
                    else:
                        edge_attr = torch.nan_to_num(edge_attr, nan=0.0, posinf=1.0, neginf=0.0)
                        if edge_attr.dim() == 1:
                            edge_attr = edge_attr.unsqueeze(1)
                        if edge_attr.shape[1] > expected_edge_dim:
                            edge_attr = edge_attr[:, :expected_edge_dim]
                        elif edge_attr.shape[1] < expected_edge_dim:
                            pad = torch.zeros((edge_attr.shape[0], expected_edge_dim - edge_attr.shape[1]), dtype=edge_attr.dtype)
                            edge_attr = torch.cat([edge_attr, pad], dim=1)
                        edge_labels = edge_attr.argmax(dim=-1).to(torch.long)

                    for edge_pos in range(edge_index.shape[1]):
                        u = int(edge_index[0, edge_pos].item())
                        v = int(edge_index[1, edge_pos].item())
                        if u < max_nodes and v < max_nodes:
                            adj_target[u, v] = 1.0
                            type_target[u, v] = int(edge_labels[edge_pos].item())

                    edge_idx = edge_index + node_offset
                    batch_edge_index_list.append(edge_idx)
                    batch_edge_attr_list.append(edge_attr)

                batch_adj_targets.append(adj_target)
                batch_type_targets.append(type_target)
                batch_indices.extend([g_idx] * max_nodes)
                node_offset += max_nodes

            X_batch = torch.cat(batch_x_list, dim=0)
            if batch_edge_index_list:
                edge_index_batch = torch.cat(batch_edge_index_list, dim=1)
                edge_attr_batch = torch.cat(batch_edge_attr_list, dim=0)
            else:
                edge_index_batch = torch.zeros((2, 0), dtype=torch.long)
                edge_attr_batch = torch.zeros((0, expected_edge_dim), dtype=torch.float32)

            batch_tensor = torch.tensor(batch_indices, dtype=torch.long)
            global_feat = standardize_feature_tensor(torch.zeros(len(batch_graphs), 2))
            batch_pockets = real_pocket.repeat(len(batch_graphs), 1)

            X_batch = X_batch.to(device)
            edge_index_batch = edge_index_batch.to(device)
            edge_attr_batch = edge_attr_batch.to(device)
            batch_tensor = batch_tensor.to(device)
            global_feat = global_feat.to(device)
            batch_pockets = batch_pockets.to(device)
            batch_adj_targets = torch.stack(batch_adj_targets).to(device)
            batch_type_targets = torch.stack(batch_type_targets).to(device)

            if not torch.isfinite(X_batch).all() or not torch.isfinite(edge_attr_batch).all():
                skipped_batches += 1
                continue

            mu, logvar = vae.encode(X_batch, edge_index_batch, edge_attr_batch, batch_tensor, global_feat)
            if not torch.isfinite(mu).all() or not torch.isfinite(logvar).all():
                skipped_batches += 1
                continue

            mu = torch.clamp(mu, -30.0, 30.0)
            logvar = torch.clamp(logvar, -20.0, 20.0)

            z = vae.reparameterize(mu, logvar)
            node_logits, edge_adj_logits, edge_type_logits = vae.decode(z, pocket_embedding=batch_pockets)

            if not torch.isfinite(node_logits).all() or not torch.isfinite(edge_adj_logits).all() or not torch.isfinite(edge_type_logits).all():
                skipped_batches += 1
                continue

            X_reshaped = X_batch.view(len(batch_graphs), max_nodes, -1)
            loss_recon = criterion_recon(node_logits, X_reshaped)
            kld = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

            atom_class_logits = atom_classifier(node_logits)
            atomic_nums = X_reshaped[..., 0].long().clamp_min(0)
            atom_labels = torch.zeros_like(atomic_nums)
            atom_labels[atomic_nums == 6] = 0
            atom_labels[atomic_nums == 7] = 1
            atom_labels[atomic_nums == 8] = 2
            atom_labels[atomic_nums == 9] = 3
            atom_labels[atomic_nums == 16] = 4
            atom_labels[atomic_nums == 17] = 5

            valid_mask = atomic_nums > 0
            if valid_mask.any():
                ce_loss = F.cross_entropy(atom_class_logits[valid_mask], atom_labels[valid_mask], weight=class_weights)
            else:
                ce_loss = torch.tensor(0.0, device=device)

            edge_adj_loss = edge_adj_criterion(edge_adj_logits, batch_adj_targets)
            edge_type_loss = F.cross_entropy(
                edge_type_logits.reshape(-1, edge_type_logits.size(-1)),
                batch_type_targets.reshape(-1),
                ignore_index=-100,
            )
            edge_loss = edge_adj_loss + edge_type_loss

            gumbel_atom = F.gumbel_softmax(atom_class_logits, tau=tau, hard=True, dim=-1)
            halogen_prob = gumbel_atom[..., halogen_idxs].sum(dim=-1)
            halogen_penalty = F.relu(halogen_prob.sum(dim=-1) - 1.0).mean()

            loss = (
                loss_recon
                + beta * kld
                + alpha_atom * ce_loss
                + lambda_edge * edge_loss
                + lambda_halogen * halogen_penalty
            )

            if torch.isnan(loss) or not torch.isfinite(loss):
                skipped_batches += 1
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            optimizer.step()

            bsz = len(batch_graphs)
            total_loss += float(loss.item()) * bsz
            total_ce += float(ce_loss.item()) * bsz
            total_halogen_pen += float(halogen_penalty.item()) * bsz
            total_edge_adj += float(edge_adj_loss.item()) * bsz
            total_edge_type += float(edge_type_loss.item()) * bsz
            n_processed += bsz

        avg_loss = total_loss / max(n_processed, 1)
        avg_ce = total_ce / max(n_processed, 1)
        avg_hal = total_halogen_pen / max(n_processed, 1)
        avg_edge_adj = total_edge_adj / max(n_processed, 1)
        avg_edge_type = total_edge_type / max(n_processed, 1)
        print(
            f'Epoch {ep+1}/{epochs_ft} - loss={avg_loss:.4f}, ce={avg_ce:.4f}, edge_adj={avg_edge_adj:.4f}, edge_type={avg_edge_type:.4f}, hal={avg_hal:.4f}, lr={optimizer.param_groups[0]["lr"]:.2e}, tau={tau:.3f}, skipped={skipped_batches}'
        )

    models_dir = ROOT / 'models'
    models_dir.mkdir(exist_ok=True, parents=True)
    torch.save(vae.state_dict(), models_dir / 'egfr_finetuned_graphvae.pth')
    print(f'\nFine-tuned VAE saved: {models_dir / "egfr_finetuned_graphvae.pth"}')


PHASE 2: Fine-tuning on EGFR (With Pocket Conditioning)
Using device for fine-tuning: cuda
Loaded 19846 EGFR graphs from graph_data.pt
Atom class counts: [492160.0, 110574.0, 45722.0, 12614.0, 4404.0, 7876.0]
Atom class weights (boosted): [0.026146795600652695, 0.5818911790847778, 2.8144888877868652, 1.020168662071228, 1.5, 1.5]
Using pocket embedding shape: (1, 128)
Loaded pretrained weights from zinc_pretrained_graphvae.pth with pocket_dim=128

Fine-tuning for 5 epochs on 19846 EGFR graphs...
Expected edge feature dim: 5
Epoch 1/5 - loss=8.8548, ce=1.5313, edge_adj=0.2342, edge_type=1.0485, hal=3.5605, lr=2.00e-04, tau=0.900, skipped=0
Epoch 2/5 - loss=7.2721, ce=1.3056, edge_adj=0.0571, edge_type=0.7837, hal=0.9391, lr=4.00e-04, tau=0.800, skipped=0
Epoch 3/5 - loss=7.1864, ce=1.2891, edge_adj=0.0560, edge_type=0.7726, hal=0.8171, lr=6.00e-04, tau=0.700, skipped=0
Epoch 4/5 - loss=7.1356, ce=1.2834, edge_adj=0.0559, edge_type=0.7706, hal=0.7888, lr=8.00e-04, tau=0.600, skipped=0
Ep

In [4]:

# ===== PHASE 3: GENERATIVE SAMPLING =====

print('\n' + '=' * 60)
print('PHASE 3: Size-Expanded Generative Sweep + GNN Scoring')
print('=' * 60)

import json
import inspect
from collections import defaultdict
from pathlib import Path
import numpy as np
import torch
import torch.nn as F
from rdkit import Chem
from torch_geometric.data import Data as PyGData

from core.gnn_architecture import EGFR_GNN_Regressor
from core.pocket_extractor import get_pocket_embedding
from core.vae_architecture import GraphVAE

# Coordinate global runtime parameters
if 'ROOT' not in globals():
    ROOT = Path.cwd().parent
if 'device' not in globals():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if 'DATA_RAW' not in globals():
    DATA_RAW = ROOT / 'data' / 'raw'
if 'DATA_PROC' not in globals():
    DATA_PROC = ROOT / 'data' / 'processed'


def load_graph_bundle(graph_file):
    loaded = torch.load(graph_file, map_location='cpu', weights_only=False)
    if isinstance(loaded, dict):
        for key in ('graphs', 'egfr_graphs', 'train_graphs', 'data'):
            candidate = loaded.get(key)
            if isinstance(candidate, (list, tuple)) and len(candidate) > 0:
                return list(candidate), loaded.get('metadata', {})
        for candidate in loaded.values():
            if isinstance(candidate, (list, tuple)) and len(candidate) > 0:
                return list(candidate), loaded.get('metadata', {})
    if isinstance(loaded, (list, tuple)):
        return list(loaded), {}
    return [], {}


def normalize_edge_index(data):
    edge_index = getattr(data, 'edge_index', None)
    if edge_index is None:
        data.edge_index = torch.zeros((2, 0), dtype=torch.long)
        return data
    if not hasattr(edge_index, 'ndim') or edge_index.ndim != 2:
        data.edge_index = torch.zeros((2, 0), dtype=torch.long)
        return data
    if edge_index.shape[0] == 2:
        return data
    if edge_index.shape[1] == 2:
        data.edge_index = edge_index.t().contiguous()
        return data
    data.edge_index = torch.zeros((2, 0), dtype=torch.long)
    return data


def safe_three_way_scaffold_split(graph_list, train_frac=0.70, val_frac=0.15):
    scaffold_to_indices = defaultdict(list)
    for idx, graph in enumerate(graph_list):
        if hasattr(graph, 'scaffold_id') and graph.scaffold_id is not None:
            scaffold_id = int(graph.scaffold_id.view(-1)[0].item())
        else:
            scaffold_id = -1
        scaffold_to_indices[scaffold_id].append(idx)

    ordered_scaffolds = sorted(scaffold_to_indices.items(), key=lambda item: (len(item[1]), item[0]))
    total = len(graph_list)
    train_target = max(1, int(round(total * train_frac)))
    val_target = max(1, int(round(total * val_frac)))
    train_cut = train_target
    val_cut = train_target + val_target

    train_idx = []
    val_idx = []
    test_idx = []
    cumulative = 0
    for scaffold_id, indices in ordered_scaffolds:
        if cumulative < train_cut:
            train_idx.extend(indices)
        elif cumulative < val_cut:
            val_idx.extend(indices)
        else:
            test_idx.extend(indices)
        cumulative += len(indices)

    return np.asarray(train_idx, dtype=int), np.asarray(val_idx, dtype=int), np.asarray(test_idx, dtype=int)


def compute_train_target_stats(graph_list):
    train_idx, _, _ = safe_three_way_scaffold_split(graph_list)
    if len(train_idx) == 0:
        return 0.0, 1.0
    y_train = []
    for idx in train_idx:
        graph = graph_list[int(idx)]
        if hasattr(graph, 'y') and graph.y is not None:
            y_train.append(float(graph.y.view(-1)[0].item()))
    if len(y_train) == 0:
        return 0.0, 1.0
    y_train = np.asarray(y_train, dtype=np.float32)
    return float(y_train.mean()), float(y_train.std() if y_train.std() > 1e-8 else 1.0)


def load_pocket_embedding_with_fallback(pdb_name, ligand_codes, embedding_dim=128):
    pdb_file = DATA_RAW / 'pdb' / pdb_name
    last_error = None
    for ligand_code in ligand_codes:
        try:
            pocket_array = get_pocket_embedding(
                str(pdb_file),
                ligand_code=ligand_code,
                pocket_radius=7.0,
                embedding_dim=embedding_dim,
            )
            pocket_tensor = torch.tensor(pocket_array, dtype=torch.float32, device=device).unsqueeze(0)
            return pocket_tensor, ligand_code
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f'Could not load pocket embedding for {pdb_name}: {last_error}')


def sample_generation_threshold():
    sampled = 0.35 + 0.08 * torch.randn(1, device=device)
    return float(torch.clamp(sampled, 0.20, 0.55).item())


def sample_target_nodes():
    sampled = 25.0 + 5.0 * torch.randn(1, device=device)
    return int(torch.clamp(sampled.round(), 18, 38).item())


def sample_target_degree():
    sampled = 2.4 + 0.12 * torch.randn(1, device=device)
    return float(torch.clamp(sampled, 2.2, 2.6).item())


def get_atom_symbol(node_logits, cl_index, carbon_index, atom_temperature=0.8, chlorine_penalty=8.0):
    logits = node_logits[:6].clone()
    logits[cl_index] -= chlorine_penalty
    logits[3] -= 1.5
    probs = torch.softmax(logits / max(atom_temperature, 1e-6), dim=0)
    sampled_idx = int(torch.multinomial(probs, num_samples=1).item())
    atom_map = ['C', 'N', 'O', 'F', 'S', 'Cl']
    return atom_map[min(sampled_idx, len(atom_map) - 1)]


def bond_order_from_logits(bond_logits, tau=0.8):
    bond_type_sample = torch.nn.functional.gumbel_softmax(bond_logits.unsqueeze(0), tau=tau, hard=True, dim=-1)[0]
    return int(bond_type_sample.argmax().item())


def atom_order_for_symbol(symbol):
    max_valence = {
        'C': 4.0,
        'N': 3.0,
        'O': 2.0,
        'F': 1.0,
        'S': 6.0,
        'Cl': 1.0,
    }
    return max_valence.get(symbol, 4.0)


def bond_type_to_rdkit(bond_type_idx):
    bond_map = {
        0: Chem.rdchem.BondType.SINGLE,
        1: Chem.rdchem.BondType.DOUBLE,
        2: Chem.rdchem.BondType.TRIPLE,
        3: Chem.rdchem.BondType.AROMATIC,
    }
    return bond_map.get(int(bond_type_idx), Chem.rdchem.BondType.SINGLE)


def build_candidate_graph(node_logits, edge_adj_logits, edge_type_logits, target_nodes, target_degree):
    atom_temperature = 0.8
    chlorine_penalty = 8.0
    atom_map = ['C', 'N', 'O', 'F', 'S', 'Cl']
    cl_index = atom_map.index('Cl')
    carbon_index = atom_map.index('C')

    node_logits = node_logits.detach().clone().cpu()
    edge_adj_logits = edge_adj_logits.detach().clone().cpu()
    edge_type_logits = edge_type_logits.detach().clone().cpu()

    node_scores = node_logits[:, :6].abs().sum(dim=-1)
    node_probs = torch.softmax(node_scores / max(atom_temperature, 1e-6), dim=0)
    target_nodes = min(target_nodes, node_logits.size(0))
    active_indices = torch.multinomial(node_probs, num_samples=target_nodes, replacement=False).sort().values
    active_node_features = node_logits[active_indices].clone()

    atom_logits = active_node_features[:, :6].clone()
    atom_logits[:, cl_index] -= chlorine_penalty
    atom_logits[:, 3] -= 1.5
    atom_probs = torch.softmax(atom_logits / max(atom_temperature, 1e-6), dim=-1)
    atom_indices = torch.multinomial(atom_probs, num_samples=1).view(-1)
    atom_symbols = [atom_map[int(idx)] for idx in atom_indices.tolist()]
    if atom_symbols and not any(symbol == 'C' for symbol in atom_symbols):
        carbon_slot = int(torch.argmax(atom_logits[:, carbon_index]).item())
        atom_symbols[carbon_slot] = 'C'

    active_adj_probs = torch.sigmoid(edge_adj_logits)[active_indices][:, active_indices]
    candidate_edges = []
    for i in range(target_nodes):
        for j in range(i + 1, target_nodes):
            candidate_edges.append((float(active_adj_probs[i, j].item()), i, j))
    candidate_edges.sort(key=lambda item: item[0], reverse=True)

    target_bonds = int(round(target_nodes * target_degree / 2.0))
    target_bonds = max(target_nodes - 1, target_bonds)
    target_bonds = min(target_bonds, max(1, len(candidate_edges)))
    threshold = sample_generation_threshold()

    filtered_edges = [item for item in candidate_edges if item[0] >= threshold]
    if len(filtered_edges) < target_bonds:
        selected_edges = candidate_edges[:target_bonds]
    else:
        selected_edges = filtered_edges[:target_bonds]

    edge_index_pairs = []
    edge_attr_pairs = []
    for _, i, j in selected_edges:
        bond_logits = edge_type_logits[active_indices[i], active_indices[j]]
        bond_type_idx = bond_order_from_logits(bond_logits, tau=atom_temperature)
        one_hot = [1.0 if k == bond_type_idx else 0.0 for k in range(edge_type_logits.size(-1))]
        edge_index_pairs.extend([[i, j], [j, i]])
        edge_attr_pairs.extend([one_hot, one_hot])

    if edge_index_pairs:
        edge_index = torch.tensor(edge_index_pairs, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attr_pairs, dtype=torch.float32)
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, edge_type_logits.size(-1)), dtype=torch.float32)

    pyg_graph = PyGData(
        x=active_node_features,
        edge_index=edge_index,
        edge_attr=edge_attr,
        global_features=torch.zeros((1, 20), dtype=torch.float32),
        pocket_embedding=torch.zeros((1, 128), dtype=torch.float32),
    )
    return pyg_graph, active_indices, selected_edges, atom_symbols, threshold


def build_rdkit_smiles(atom_symbols, selected_edges):
    mol = Chem.RWMol()
    for symbol in atom_symbols:
        atom = Chem.Atom(symbol)
        atom.SetFormalCharge(0)
        atom.SetNoImplicit(False)
        atom.SetNumExplicitHs(0)
        mol.AddAtom(atom)

    current_valence = [0.0 for _ in atom_symbols]
    for _, i, j in selected_edges:
        bond_order_idx = 0
        bond_order = 1.0
        if current_valence[i] + bond_order <= atom_order_for_symbol(atom_symbols[i]) and current_valence[j] + bond_order <= atom_order_for_symbol(atom_symbols[j]):
            try:
                mol.AddBond(int(i), int(j), bond_type_to_rdkit(bond_order_idx))
                current_valence[i] += bond_order
                current_valence[j] += bond_order
            except Exception:
                pass

    rdkit_mol = mol.GetMol()
    try:
        rdkit_mol.UpdatePropertyCache(strict=False)
    except Exception:
        pass
    try:
        Chem.SanitizeMol(rdkit_mol)
        smiles = Chem.MolToSmiles(rdkit_mol, canonical=True)
        return rdkit_mol, smiles
    except Exception:
        return None, None


def score_graph(model, graph, pocket_embedding, y_mean, y_std, current_pocket_dim=128):
    batch = torch.zeros(graph.x.size(0), dtype=torch.long, device=device)
    global_features = torch.zeros((1, 20), dtype=torch.float32, device=device)
    
    if pocket_embedding.size(-1) != current_pocket_dim:
        if current_pocket_dim < pocket_embedding.size(-1):
            adjusted_pocket = pocket_embedding[..., :current_pocket_dim]
        else:
            adjusted_pocket = torch.zeros((1, current_pocket_dim), dtype=torch.float32, device=device)
            adjusted_pocket[..., :pocket_embedding.size(-1)] = pocket_embedding
    else:
        adjusted_pocket = pocket_embedding

    preds = model(
        graph.x.to(device),
        graph.edge_index.to(device),
        batch,
        edge_attr=graph.edge_attr.to(device),
        global_features=global_features,
        pos=None,
        fingerprint=None,
        pocket_embedding=adjusted_pocket.to(device),
    ).view(-1)
    if y_std > 0:
        preds = preds * y_std + y_mean
    return float(preds.item())


# ===== INITIALIZATION AND MODEL CONTEXT ASSEMBLY =====

# Setup foundational generator configurations dynamically
vae_sig = inspect.signature(GraphVAE.__init__)
vae_params = vae_sig.parameters

def create_vae_instance(target_latent_dim=128, target_pocket_dim=128):
    vae_kwargs = {}
    if 'max_nodes' in vae_params: vae_kwargs['max_nodes'] = 50
    elif 'num_nodes' in vae_params: vae_kwargs['num_nodes'] = 50

    if 'atom_features' in vae_params: vae_kwargs['atom_features'] = 8
    elif 'node_features' in vae_params: vae_kwargs['node_features'] = 8
    elif 'in_channels' in vae_params: vae_kwargs['in_channels'] = 8
    elif 'num_features' in vae_params: vae_kwargs['num_features'] = 8

    if 'edge_features' in vae_params: vae_kwargs['edge_features'] = 5
    elif 'edge_dim' in vae_params: vae_kwargs['edge_dim'] = 5
    elif 'num_edges' in vae_params: vae_kwargs['num_edges'] = 5

    if 'latent_dim' in vae_params: vae_kwargs['latent_dim'] = target_latent_dim
    if 'pocket_dim' in vae_params: vae_kwargs['pocket_dim'] = target_pocket_dim
    return GraphVAE(**vae_kwargs).to(device)


# Initialize high-capacity 128D structural shell by default
vae = create_vae_instance(target_latent_dim=128, target_pocket_dim=128)
active_vae_pocket_dim = 128
active_vae_latent_dim = 128

fine_tuned_path = ROOT / 'models' / 'egfr_finetuned_graphvae.pth'
if fine_tuned_path.exists():
    try:
        loaded_state = torch.load(fine_tuned_path, map_location=device)
        vae.load_state_dict(loaded_state)
        print(f'✓ Loaded fine-tuned high-capacity weights from {fine_tuned_path.name}')
    except RuntimeError as err:
        print(f'⚠️ Generative VAE size mismatch detected: {err}')
        print('-> Falling back to legacy network layout architecture (latent_dim=64, pocket_dim=128)...')
        
        # Free memory and re-instantiate using exact matching checkpoint dimensions
        del vae
        active_vae_pocket_dim = 128
        active_vae_latent_dim = 64
        vae = create_vae_instance(target_latent_dim=64, target_pocket_dim=128)
        loaded_state = torch.load(fine_tuned_path, map_location=device)
        vae.load_state_dict(loaded_state)
        print(f'✓ Successfully bound fine-tuned VAE from matching checkpoint weights.')
else:
    print(f'Fine-tuned checkpoint not found at {fine_tuned_path}; using current in-memory weights')

vae.eval()

# Always retrieve complete 128D physical crystal vectors from data pipeline
pocket_dim = 128
primary_pocket, primary_pocket_code = load_pocket_embedding_with_fallback('3W2S.pdb', ['GFT', 'W2R'], embedding_dim=pocket_dim)
secondary_pocket, secondary_pocket_code = load_pocket_embedding_with_fallback('6LUD.pdb', ['9KC', 'YY3'], embedding_dim=pocket_dim)
print(f'✓ Loaded pocket embeddings: 3W2S={tuple(primary_pocket.shape)} via {primary_pocket_code}, 6LUD={tuple(secondary_pocket.shape)} via {secondary_pocket_code}')

graph_file = DATA_PROC / 'graph_data.pt'
processed_graphs, metadata = load_graph_bundle(graph_file)
processed_graphs = [normalize_edge_index(g) for g in processed_graphs if hasattr(g, 'x')]
y_mean, y_std = compute_train_target_stats(processed_graphs)
print(f'✓ Reconstructed train target scaling: mean={y_mean:.4f}, std={y_std:.4f}')

n_samples = 500
latent_dim = active_vae_latent_dim
results_dir = ROOT / 'results' / 'generated_mols'
results_dir.mkdir(parents=True, exist_ok=True)
models_dir = ROOT / 'models'

print(f'\nGenerating {n_samples} molecules with dynamic size sampling...\n')

# Load GNN Assessor with Dynamic State-Dict Fallback Handling
gnn_model_path = models_dir / 'best_gnn_regressor.pth'
gnn_target_pocket_dim = 128

# First try: high capacity model
gnn_model = EGFR_GNN_Regressor(
    node_features=8,
    edge_features=5,
    global_features=20,
    hidden_dim=128,
    output_dim=1,
    num_layers=4,
    dropout_rate=0.3,
    fingerprint_dim=1024,
    pocket_dim=128,
).to(device)

if gnn_model_path.exists():
    try:
        gnn_state = torch.load(gnn_model_path, map_location=device)
        gnn_model.load_state_dict(gnn_state)
        print(f'✓ Loaded high-capacity GNN assessor from {gnn_model_path.name}')
    except RuntimeError as err:
        print(f'⚠️ Assessor high-capacity mismatch detected: {err}')
        print('-> Falling back to exact legacy dimensions matching saved checkpoint...')
        
        del gnn_model
        gnn_target_pocket_dim = 128 
        gnn_model = EGFR_GNN_Regressor(
            node_features=8,
            edge_features=5,         # 5 + 1 (distance scalar) = 6 (matches checkpoint perfectly)
            global_features=20,
            hidden_dim=64,           # Aligned with [64, 64] shape tracking
            output_dim=1,
            num_layers=2,            # Strips out missing convs blocks
            dropout_rate=0.3,
            fingerprint_dim=1024,
            pocket_dim=128,          # Reverted to full 128D width required by checkpoint
        ).to(device)
        
        gnn_state = torch.load(gnn_model_path, map_location=device)
        gnn_model.load_state_dict(gnn_state)
        print(f'✓ Successfully bound legacy GNN assessor from {gnn_model_path.name}')
else:
    print(f'GNN assessor checkpoint not found at {gnn_model_path}; scores will be unavailable')
    gnn_model = None

if gnn_model is not None:
    gnn_model.eval()

candidate_rows = []
valid_smiles = []
passed_count = 0

# Adjust target pocket conditioning context down on-the-fly to fit the loaded decoder layout
decoder_primary_pocket = primary_pocket[..., :active_vae_pocket_dim] if active_vae_pocket_dim < 128 else primary_pocket

# ===== PRODUCTION EVALUATION ITERATION =====

with torch.no_grad():
    for i in range(n_samples):
        z_sample = torch.randn(1, latent_dim, device=device)
        node_features, edge_adj_logits, edge_type_logits = vae.decode(z_sample, pocket_embedding=decoder_primary_pocket)
        node_features = node_features[0].clone()
        edge_adj_logits = edge_adj_logits[0].clone()
        edge_type_logits = edge_type_logits[0].clone()
        
        # Apply Inference-Time Categorical Logit Mask to damp chlorines
        node_features[:, 5] -= 8.0

        target_nodes = sample_target_nodes()
        target_degree = sample_target_degree()
        pyg_graph, active_indices, selected_edges, atom_symbols, threshold = build_candidate_graph(
            node_features,
            edge_adj_logits,
            edge_type_logits,
            target_nodes,
            target_degree,
        )
        rdkit_mol, smiles = build_rdkit_smiles(atom_symbols, selected_edges)

        if smiles is None:
            continue
        if '.' in smiles:
            continue
        if not (18 <= rdkit_mol.GetNumAtoms() <= 38):
            continue

        if gnn_model is not None:
            gatekeeper_score = score_graph(gnn_model, pyg_graph, primary_pocket, y_mean, y_std, current_pocket_dim=gnn_target_pocket_dim)
            triple_mutant_score = score_graph(gnn_model, pyg_graph, secondary_pocket, y_mean, y_std, current_pocket_dim=gnn_target_pocket_dim)
            mean_score = 0.5 * (gatekeeper_score + triple_mutant_score)
        else:
            gatekeeper_score = float('nan')
            triple_mutant_score = float('nan')
            mean_score = float('nan')

        candidate_rows.append({
            'idx': i + 1,
            'smiles': smiles,
            'target_nodes': int(target_nodes),
            'active_nodes': int(active_indices.numel()),
            'selected_bonds': int(len(selected_edges)),
            'average_degree': float((2.0 * len(selected_edges)) / max(int(active_indices.numel()), 1)),
            'threshold': float(threshold),
            'gatekeeper_pIC50': gatekeeper_score,
            'triple_mutant_pIC50': triple_mutant_score,
            'mean_pIC50': mean_score,
        })
        valid_smiles.append(smiles)
        passed_count += 1

        if passed_count <= 5 or passed_count % 50 == 0:
            print(
                f"  Sample {i + 1:4d}: nodes={int(active_indices.numel())}, bonds={len(selected_edges)}, "
                f"avg_degree={(2.0 * len(selected_edges)) / max(int(active_indices.numel()), 1):.2f}, "
                f"threshold={threshold:.2f}, gatekeeper_pIC50={gatekeeper_score:.2f}, "
                f"6LUD_pIC50={triple_mutant_score:.2f}, mean_pIC50={mean_score:.2f}, smiles={smiles}"
            )

print(f'\n✓ Generated {n_samples} samples')
print(f'✓ Valid, size-expanded candidates: {passed_count} / {n_samples}')

candidate_rows = [row for row in candidate_rows if np.isfinite(row['mean_pIC50'])]
candidate_rows.sort(key=lambda row: row['mean_pIC50'], reverse=True)

print('\nTop-scoring EGFR inhibitor candidates by mean predicted pIC50:')
for rank, row in enumerate(candidate_rows[:10], start=1):
    print(
        f"  {rank:2d}. mean_pIC50={row['mean_pIC50']:.3f} | "
        f"3W2S={row['gatekeeper_pIC50']:.3f} | 6LUD={row['triple_mutant_pIC50']:.3f} | "
        f"nodes={row['active_nodes']} | bonds={row['selected_bonds']} | {row['smiles']}"
    )

summary = {
    'phase': 'VAE_Generative_Sampling_Size_Expanded_GNN_Aligned',
    'n_generated': n_samples,
    'n_valid_candidates': len(candidate_rows),
    'model_path': str(fine_tuned_path),
    'gnn_model_path': str(gnn_model_path),
    'vae_decoder_pocket_dim_used': active_vae_pocket_dim,
    'gnn_assessor_pocket_dim_used': gnn_target_pocket_dim,
    'latent_dim': latent_dim,
    'pocket_dim': pocket_dim,
    'primary_pocket': {'pdb': '3W2S.pdb', 'ligand_code': primary_pocket_code},
    'secondary_pocket': {'pdb': '6LUD.pdb', 'ligand_code': secondary_pocket_code},
    'target_mean_pIC50': y_mean,
    'target_std_pIC50': y_std,
    'samples': candidate_rows,
}
with open(results_dir / 'vae_generation_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

print(f"\nResults saved to: {results_dir / 'vae_generation_summary.json'}")
print('\n✅ Final optimized generation sweep complete!')


PHASE 3: Size-Expanded Generative Sweep + GNN Scoring
⚠️ Generative VAE size mismatch detected: Error(s) in loading state_dict for GraphVAE:
	size mismatch for encoder.to_mu.weight: copying a param with shape torch.Size([64, 130]) from checkpoint, the shape in current model is torch.Size([128, 130]).
	size mismatch for encoder.to_mu.bias: copying a param with shape torch.Size([64]) from checkpoint, the shape in current model is torch.Size([128]).
	size mismatch for encoder.to_logvar.weight: copying a param with shape torch.Size([64, 130]) from checkpoint, the shape in current model is torch.Size([128, 130]).
	size mismatch for encoder.to_logvar.bias: copying a param with shape torch.Size([64]) from checkpoint, the shape in current model is torch.Size([128]).
	size mismatch for decoder.mlp_expand.0.weight: copying a param with shape torch.Size([128, 128]) from checkpoint, the shape in current model is torch.Size([128, 192]).
-> Falling back to legacy network layout architecture (latent

In [5]:
# ===== PHASE 3 ANALYSIS =====
import json
from collections import Counter

summary_path = ROOT / 'results' / 'generated_mols' / 'vae_generation_summary.json'
with open(summary_path, 'r') as f:
    data = json.load(f)

samples = data.get('samples', [])
smiles_list = [s.get('smiles') for s in samples if s.get('smiles')]
n_total = len(samples)
n_valid_smiles = len(smiles_list)

n_contains_cl = sum('Cl' in s for s in smiles_list)
n_contains_f = sum('F' in s for s in smiles_list)
pass_quick_filter = sum(('Cl' not in s) and (sum(s.count(tok) for tok in ['Cl','Br','F','I']) <= 1) for s in smiles_list)

active_nodes = [int(s.get('active_nodes', 0)) for s in samples]
selected_bonds = [int(s.get('selected_bonds', 0)) for s in samples]
thresholds = [s.get('chosen_threshold') for s in samples]

# Token-level atom heuristic from SMILES strings
atom_counter = Counter()
for s in smiles_list:
    atom_counter['Cl'] += s.count('Cl')
    atom_counter['F'] += s.count('F')
    atom_counter['N'] += s.count('N')
    atom_counter['O'] += s.count('O')
    atom_counter['S'] += s.count('S')
    # crude carbon estimate: uppercase C not part of Cl
    atom_counter['C'] += s.count('C') - s.count('Cl')

print('='*60)
print('PHASE 3 ANALYSIS SUMMARY')
print('='*60)
print(f'Total requested samples: {n_total}')
print(f'Non-empty SMILES: {n_valid_smiles}')
print(f'Quick-filter pass count: {pass_quick_filter} / {n_total} ({(100*pass_quick_filter/max(n_total,1)):.2f}%)')
print(f'SMILES containing Cl: {n_contains_cl} / {n_valid_smiles} ({(100*n_contains_cl/max(n_valid_smiles,1)):.2f}%)')
print(f'SMILES containing F: {n_contains_f} / {n_valid_smiles} ({(100*n_contains_f/max(n_valid_smiles,1)):.2f}%)')

if active_nodes:
    print(f'Active nodes: mean={sum(active_nodes)/len(active_nodes):.2f}, min={min(active_nodes)}, max={max(active_nodes)}')
if selected_bonds:
    print(f'Selected bonds: mean={sum(selected_bonds)/len(selected_bonds):.2f}, min={min(selected_bonds)}, max={max(selected_bonds)}')

threshold_counter = Counter(thresholds)
print(f'Chosen threshold distribution: {dict(threshold_counter)}')
print(f'Approx atom token counts: {dict(atom_counter)}')

print('\nFirst 5 generated SMILES:')
for i, s in enumerate(smiles_list[:5], start=1):
    print(f'  {i}. {s}')


PHASE 3 ANALYSIS SUMMARY
Total requested samples: 123
Non-empty SMILES: 123
Quick-filter pass count: 123 / 123 (100.00%)
SMILES containing Cl: 0 / 123 (0.00%)
SMILES containing F: 6 / 123 (4.88%)
Active nodes: mean=25.47, min=18, max=32
Selected bonds: mean=30.54, min=20, max=41
Chosen threshold distribution: {None: 123}
Approx atom token counts: {'Cl': 0, 'F': 6, 'N': 51, 'O': 0, 'S': 3, 'C': 3073}

First 5 generated SMILES:
  1. CCCC1CC1CC1C2C3C4(CC4CCC4CC4CC4C(F)CC5C(C)C45)C123
  2. CCCC1CC1CC1C2C3C4(CC4CCC4CC4CC45CC4CC4C(C)C45)C123
  3. CCCC1C2C3C4(CC4CCC4CC4CC45CC4CC4C(C)C45)C123
  4. CCCC1C2C3C4(CC4CCC4CC4CC45CC4CC4C(C)C45)C123
  5. CCCC1CC1CC1C2C3C4(CC4CCC4CC4CC45CC4CC4C(C)C45)C123


In [6]:
# ===== PHASE 4: OMID'S GENERATIVE METRICS EVALUATION =====
import json
from pathlib import Path
import torch
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import DataStructs

print('=' * 60)
print("PHASE 4: EVALUATING VALIDITY, UNIQUENESS, NOVELTY & DIVERSITY")
print('=' * 60)

# 1. Setup Paths (Using your established ROOT structure)
ROOT = Path.cwd().parent
results_dir = ROOT / 'results' / 'generated_mols'
summary_path = results_dir / 'vae_generation_summary.json'
graph_file = ROOT / 'data' / 'processed' / 'graph_data.pt'

# 2. Load Generated SMILES
with open(summary_path, 'r') as f:
    gen_data = json.load(f)
generated_smiles_raw = [s.get('smiles') for s in gen_data.get('samples', []) if s.get('smiles')]

# 3. Load Training SMILES (The "Answer Key" for Novelty)
print("Loading training data to verify Novelty...")
loaded_payload = torch.load(graph_file, weights_only=False, map_location='cpu')

egfr_graphs = []
if isinstance(loaded_payload, dict):
    for key in ('graphs', 'egfr_graphs', 'train_graphs', 'data'):
        if key in loaded_payload and len(loaded_payload[key]) > 0:
            egfr_graphs = loaded_payload[key]
            break
elif isinstance(loaded_payload, (list, tuple)):
    egfr_graphs = list(loaded_payload)

train_smiles_set = set()
for g in egfr_graphs:
    if hasattr(g, 'SMILES') and g.SMILES:
        train_smiles_set.add(g.SMILES)
print(f"Loaded {len(train_smiles_set)} unique training SMILES.")

# 4. Calculate Validity
print("\nCalculating Metrics...")
valid_mols = []
valid_smiles = []
for smi in generated_smiles_raw:
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        valid_mols.append(mol)
        # Convert to canonical SMILES to ensure strict matching
        valid_smiles.append(Chem.MolToSmiles(mol, canonical=True))

validity = len(valid_smiles) / len(generated_smiles_raw) if generated_smiles_raw else 0

# 5. Calculate Uniqueness
unique_smiles_set = set(valid_smiles)
uniqueness = len(unique_smiles_set) / len(valid_smiles) if valid_smiles else 0

# 6. Calculate Novelty 
novel_smiles = [smi for smi in unique_smiles_set if smi not in train_smiles_set]
novelty = len(novel_smiles) / len(unique_smiles_set) if unique_smiles_set else 0

# 7. Calculate Diversity (Internal structural variety via Tanimoto distance)
diversity = 0.0
if len(unique_smiles_set) > 1:
    fps = [AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(smi), 2, nBits=1024) for smi in unique_smiles_set]
    similarities = []
    for i in range(len(fps)):
        for j in range(i + 1, len(fps)):
            sim = DataStructs.TanimotoSimilarity(fps[i], fps[j])
            similarities.append(sim)
    diversity = 1.0 - np.mean(similarities)

# 8. Print Final Report for Supervisor
print("\n" + "="*60)
print("GENERATIVE EVALUATION REPORT (OMID'S METRICS)")
print("="*60)
print(f"Total Generated Attempts: {len(generated_smiles_raw)}")
print(f"Validity:   {validity * 100:.2f}%  (Are they chemically possible?)")
print(f"Uniqueness: {uniqueness * 100:.2f}%  (Did it avoid repeating the same molecule?)")
print(f"Novelty:    {novelty * 100:.2f}%  (Are they brand new to the training dataset?)")
print(f"Diversity:  {diversity * 100:.2f}%  (How structurally distinct are they from each other?)")
print("="*60)

PHASE 4: EVALUATING VALIDITY, UNIQUENESS, NOVELTY & DIVERSITY
Loading training data to verify Novelty...
Loaded 19846 unique training SMILES.

Calculating Metrics...

GENERATIVE EVALUATION REPORT (OMID'S METRICS)
Total Generated Attempts: 123
Validity:   100.00%  (Are they chemically possible?)
Uniqueness: 77.24%  (Did it avoid repeating the same molecule?)
Novelty:    100.00%  (Are they brand new to the training dataset?)
Diversity:  63.68%  (How structurally distinct are they from each other?)


[14:02:37] DEPRECATION WARNING: please use MorganGenerator
[14:02:37] DEPRECATION WARNING: please use MorganGenerator
[14:02:37] DEPRECATION WARNING: please use MorganGenerator
[14:02:37] DEPRECATION WARNING: please use MorganGenerator
[14:02:37] DEPRECATION WARNING: please use MorganGenerator
[14:02:37] DEPRECATION WARNING: please use MorganGenerator
[14:02:37] DEPRECATION WARNING: please use MorganGenerator
[14:02:37] DEPRECATION WARNING: please use MorganGenerator
[14:02:37] DEPRECATION WARNING: please use MorganGenerator
[14:02:37] DEPRECATION WARNING: please use MorganGenerator
[14:02:37] DEPRECATION WARNING: please use MorganGenerator
[14:02:37] DEPRECATION WARNING: please use MorganGenerator
[14:02:37] DEPRECATION WARNING: please use MorganGenerator
[14:02:37] DEPRECATION WARNING: please use MorganGenerator
[14:02:37] DEPRECATION WARNING: please use MorganGenerator
[14:02:37] DEPRECATION WARNING: please use MorganGenerator
[14:02:37] DEPRECATION WARNING: please use MorganGenerat

In [7]:
# ===== PHASE 5: PHARMACOPHORE FILTERING (MET793 HINGE RULE) =====
import json
from pathlib import Path
from rdkit import Chem

print('=' * 60)
print("PHASE 5: BIOLOGICAL PHARMACOPHORE FILTERING")
print('=' * 60)

# 1. Setup Paths
ROOT = Path.cwd().parent
results_dir = ROOT / 'results' / 'generated_mols'
summary_path = results_dir / 'vae_generation_summary.json'
filtered_path = results_dir / 'pharmacophore_filtered_candidates.json'

# 2. Load the generated batch
try:
    with open(summary_path, 'r') as f:
        data = json.load(f)
    samples = data.get('samples', [])
    print(f"Loaded {len(samples)} raw candidates from generation sweep.")
except FileNotFoundError:
    print(f"Error: Could not find {summary_path}. Run Phase 3 generation first!")
    samples = []

# 3. Apply the Met793 Hinge Rule
# Rule: Must have >= 2 Nitrogens, OR (>= 1 Nitrogen AND >= 1 Oxygen)
survivors = []

for candidate in samples:
    smi = candidate.get('smiles')
    if not smi: 
        continue
        
    mol = Chem.MolFromSmiles(smi)
    if mol is None: 
        continue
        
    # Count heteroatoms strictly by atomic number
    n_count = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 7)
    o_count = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 8)
    f_count = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 9)
    
    # The Pharmacophore Logic Gate
    if n_count >= 2 or (n_count >= 1 and o_count >= 1):
        candidate['N_count'] = n_count
        candidate['O_count'] = o_count
        candidate['F_count'] = f_count
        survivors.append(candidate)

# 4. Sort the survivors by GNN predicted affinity
survivors.sort(key=lambda x: x.get('mean_pIC50', -999), reverse=True)

# 5. Save the filtered list
data['filtered_samples'] = survivors
data['n_filtered_candidates'] = len(survivors)

with open(filtered_path, 'w') as f:
    json.dump(data, f, indent=2)

# 6. Print the Results for the Dissertation
print(f"\nFiltering Complete. Eliminated {len(samples) - len(survivors)} pure carbon/weak-binding scaffolds.")
print(f"Total Viable Hinge-Binding Candidates: {len(survivors)}\n")

if len(survivors) > 0:
    print("🏆 TOP 5 BIOLOGICALLY VIABLE CANDIDATES (Ranked by GNN pIC50):")
    for i, row in enumerate(survivors[:5], 1):
        print(f"  {i}. pIC50={row.get('mean_pIC50'):.3f} | N:{row['N_count']} O:{row['O_count']} F:{row['F_count']} | {row['smiles']}")
else:
    print("⚠️ No candidates survived the strict pharmacophore filter.")
    print("Recommendation: Run Phase 3 again with n_samples = 2000 to increase the pool size.")

PHASE 5: BIOLOGICAL PHARMACOPHORE FILTERING
Loaded 123 raw candidates from generation sweep.

Filtering Complete. Eliminated 114 pure carbon/weak-binding scaffolds.
Total Viable Hinge-Binding Candidates: 9

🏆 TOP 5 BIOLOGICALLY VIABLE CANDIDATES (Ranked by GNN pIC50):
  1. pIC50=-3.074 | N:2 O:0 F:0 | CCCC1C2C3C(CCCCN4C5C(C67CC6CC6C(C)C67)N54)C123
  2. pIC50=-3.089 | N:2 O:0 F:0 | CCC1CC1CCC1C2C1C21CC1CCC1CC1NC1NCCC2C(C)C12
  3. pIC50=-3.089 | N:2 O:0 F:0 | CCNC1C2N1C21CC1CCC1CC1CC12CC1CC1C(C)C12
  4. pIC50=-3.094 | N:2 O:0 F:0 | CCC1C2C1N2C1CC1CCCC1CC12CC2CNC1CC1CC1CCCC2C(C)C12
  5. pIC50=-3.095 | N:2 O:0 F:0 | CC1C2CCCC(CNCCCCC3CC34NC4CCCCCCC3CC3)C12
